In [1]:
print("Hello, flu project!")

Hello, flu project!


In [3]:
import os

# Show the folder Python is looking in
print(os.getcwd())

# Show the files inside that folder
print(os.listdir())

c:\Users\Emirhan\Desktop\flu_project
['cvs-files', 'flu_notebook.ipynb']


In [4]:
import os

# Show the files inside the cvs-files folder
print(os.listdir("cvs-files"))

['submission_format.csv', 'test_set_features.csv', 'training_set_features.csv', 'training_set_labels.csv']


In [5]:
import pandas as pd

# Read the features file from the cvs-files folder
features = pd.read_csv("cvs-files/training_set_features.csv", index_col="respondent_id")

# Print the number of rows and columns
print(features.shape)

(26707, 35)


In [7]:
# Read the labels file (the correct answers)
labels = pd.read_csv("cvs-files/training_set_labels.csv", index_col="respondent_id")

# Print the number of rows and columns
print(labels.shape)

(26707, 2)


In [9]:
# Show the first 5 rows of the labels table
print(labels.head())

# Show how many people got each vaccine (the average of 0/1 values)
print(labels.mean())

               h1n1_vaccine  seasonal_vaccine
respondent_id                                
0                         0                 0
1                         0                 1
2                         0                 0
3                         0                 1
4                         0                 0
h1n1_vaccine        0.212454
seasonal_vaccine    0.465608
dtype: float64


In [11]:
# Calculate the percentage of empty cells in each column
missing_percent = features.isnull().mean() * 100

# Show the 10 columns with the most empty cells
print(missing_percent.sort_values(ascending=False).head(10).round(1))

employment_occupation    50.4
employment_industry      49.9
health_insurance         46.0
income_poverty           16.6
doctor_recc_seasonal      8.1
doctor_recc_h1n1          8.1
rent_or_own               7.6
employment_status         5.5
marital_status            5.3
education                 5.3
dtype: float64


In [13]:
# Her sütunun veri türünü göster
print(features.dtypes)

# %%
# Kaç sütun sayı, kaç sütun yazı?
print(features.dtypes.value_counts())

h1n1_concern                   float64
h1n1_knowledge                 float64
behavioral_antiviral_meds      float64
behavioral_avoidance           float64
behavioral_face_mask           float64
behavioral_wash_hands          float64
behavioral_large_gatherings    float64
behavioral_outside_home        float64
behavioral_touch_face          float64
doctor_recc_h1n1               float64
doctor_recc_seasonal           float64
chronic_med_condition          float64
child_under_6_months           float64
health_worker                  float64
health_insurance               float64
opinion_h1n1_vacc_effective    float64
opinion_h1n1_risk              float64
opinion_h1n1_sick_from_vacc    float64
opinion_seas_vacc_effective    float64
opinion_seas_risk              float64
opinion_seas_sick_from_vacc    float64
age_group                          str
education                          str
race                               str
sex                                str
income_poverty           

In [16]:
# Get the names of the numeric columns
numeric_cols = features.select_dtypes(include="number").columns.tolist()

# Get the names of the text (non-numeric) columns
text_cols = features.select_dtypes(exclude="number").columns.tolist()

# Check how many columns of each type we have
print("Number of numeric columns:", len(numeric_cols))
print("Number of text columns:", len(text_cols))
print(text_cols)

Number of numeric columns: 23
Number of text columns: 12
['age_group', 'education', 'race', 'sex', 'income_poverty', 'marital_status', 'rent_or_own', 'employment_status', 'hhs_geo_region', 'census_msa', 'employment_industry', 'employment_occupation']


In [18]:
from sklearn.model_selection import train_test_split

# Split the data: 80% for training, 20% for validation
X_train, X_val, y_train, y_val = train_test_split(
    features,        # the survey answers
    labels,          # the correct answers (did the person get the vaccine?)
    test_size=0.2,   # keep 20% aside for validation
    shuffle=True,    # mix the rows randomly before splitting
    random_state=6   # makes the split the same every time we run it
)

# Check the sizes
print("Training features:", X_train.shape)
print("Training labels:", y_train.shape)
print("Validation features:", X_val.shape)
print("Validation labels:", y_val.shape)

Training features: (21365, 35)
Training labels: (21365, 2)
Validation features: (5342, 35)
Validation labels: (5342, 2)


In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Steps for the numeric columns: fill empty cells, then rescale
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler()),
])

# Steps for the text columns: fill empty cells, then turn categories into 0/1 columns
text_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

# Apply the right steps to the right columns
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("text", text_pipeline, text_cols),
])

# Learn the rules from the training data only, then apply them to both sets
X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared = preprocessor.transform(X_val)

# Check the new shapes
print("Prepared training features:", X_train_prepared.shape)
print("Prepared validation features:", X_val_prepared.shape)

Prepared training features: (21365, 112)
Prepared validation features: (5342, 112)


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import roc_auc_score

# Create the model: one logistic regression per target
model = MultiOutputClassifier(LogisticRegression(max_iter=1000))

# Train the model on the prepared training data
model.fit(X_train_prepared, y_train)

# Predict probabilities for the validation set
# predict_proba returns one table per target: [no, yes] probabilities for each person
val_probs = model.predict_proba(X_val_prepared)

# Take only the "yes" probability (column 1) for each target
h1n1_probs = val_probs[0][:, 1]
seasonal_probs = val_probs[1][:, 1]

# Calculate the ROC AUC score for each target
h1n1_auc = roc_auc_score(y_val["h1n1_vaccine"], h1n1_probs)
seasonal_auc = roc_auc_score(y_val["seasonal_vaccine"], seasonal_probs)

# The competition score is the mean of the two
print("H1N1 AUC:", round(h1n1_auc, 4))
print("Seasonal AUC:", round(seasonal_auc, 4))
print("Mean AUC:", round((h1n1_auc + seasonal_auc) / 2, 4))

H1N1 AUC: 0.8401
Seasonal AUC: 0.8611
Mean AUC: 0.8506


In [26]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Some models need a regular (dense) table instead of a compact (sparse) one
# This converts the data only if needed
def to_dense(data):
    return data.toarray() if hasattr(data, "toarray") else data

X_train_dense = to_dense(X_train_prepared)
X_val_dense = to_dense(X_val_prepared)

# Create the model: one gradient boosting model per target
gb_model = MultiOutputClassifier(
    HistGradientBoostingClassifier(
        learning_rate=0.05,   # small steps so it learns carefully
        max_iter=200,         # number of small trees to build
        random_state=6        # same results every time we run it
    )
)

# Train the model on the prepared training data
gb_model.fit(X_train_dense, y_train)

# Predict probabilities for the validation set
gb_probs = gb_model.predict_proba(X_val_dense)

# Take only the "yes" probability for each target
gb_h1n1_probs = gb_probs[0][:, 1]
gb_seasonal_probs = gb_probs[1][:, 1]

# Calculate the ROC AUC score for each target
gb_h1n1_auc = roc_auc_score(y_val["h1n1_vaccine"], gb_h1n1_probs)
gb_seasonal_auc = roc_auc_score(y_val["seasonal_vaccine"], gb_seasonal_probs)

print("Gradient boosting H1N1 AUC:", round(gb_h1n1_auc, 4))
print("Gradient boosting Seasonal AUC:", round(gb_seasonal_auc, 4))
print("Gradient boosting Mean AUC:", round((gb_h1n1_auc + gb_seasonal_auc) / 2, 4))

Gradient boosting H1N1 AUC: 0.8475
Gradient boosting Seasonal AUC: 0.8679
Gradient boosting Mean AUC: 0.8577


In [28]:
import os

# Check that the test files are in the folder
print(os.listdir("cvs-files"))

['submission_format.csv', 'test_set_features.csv', 'training_set_features.csv', 'training_set_labels.csv']


In [30]:
# Read the test features and the submission template
test_features = pd.read_csv("cvs-files/test_set_features.csv", index_col="respondent_id")
submission = pd.read_csv("cvs-files/submission_format.csv", index_col="respondent_id")

print("Test features:", test_features.shape)
print("Submission template:", submission.shape)

# Learn the preprocessing rules from ALL the training data this time
X_all_prepared = to_dense(preprocessor.fit_transform(features))

# Apply the same rules to the test data
X_test_prepared = to_dense(preprocessor.transform(test_features))

# Train the final model on ALL the training data
final_model = MultiOutputClassifier(
    HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=200,
        random_state=6
    )
)
final_model.fit(X_all_prepared, labels)

# Predict probabilities for the test set
test_probs = final_model.predict_proba(X_test_prepared)

# Fill the submission template with the "yes" probabilities
submission["h1n1_vaccine"] = test_probs[0][:, 1]
submission["seasonal_vaccine"] = test_probs[1][:, 1]

# Save the file
submission.to_csv("my_submission.csv")

# Check the first rows
print(submission.head())

Test features: (26708, 35)
Submission template: (26708, 2)
               h1n1_vaccine  seasonal_vaccine
respondent_id                                
26707              0.081300          0.192340
26708              0.038069          0.041231
26709              0.490552          0.743665
26710              0.630120          0.879955
26711              0.265621          0.521872


In [33]:
import os

# Show the folder where my_submission.csv was saved
print(os.getcwd())
print("my_submission.csv" in os.listdir())

c:\Users\Emirhan\Desktop\flu_project
True
